In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, datediff
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

print("Bibliotecas importadas")

Bibliotecas importadas


In [3]:
spark = SparkSession.builder \
        .appName("classic-ml") \
        .getOrCreate()

print("Spark iniciado")

Spark iniciado


In [4]:
df = spark.read.csv(
    "olist_orders_dataset.csv",
    header=True,
    inferSchema=True
)

print("CSV LIDO")

CSV LIDO


In [6]:
df = df.withColumn(
    "atrasado",
    (col("order_delivered_customer_date") > col("order_estimated_delivery_date")).cast("int")
)

print("Nova label criada")

Nova label criada


In [10]:
# Engenharia de Feature

df = df.withColumn(
    "dias_estimados",
    datediff(col("order_estimated_delivery_date"), col("order_purchase_timestamp"))
)

df = df.withColumn(
    "dias_aprovacao",
    datediff(col("order_approved_at"), col("order_purchase_timestamp"))
)

df = df.withColumn(
    "dias_envio",
    datediff(col("order_delivered_carrier_date"), col("order_purchase_timestamp"))
)

df = df.select(
    "dias_estimados",
    "dias_aprovacao",
    "dias_envio",
    "atrasado"
).dropna()

print("Engenharia de Feature concluída")

Engenharia de Feature concluída


In [11]:
# Vetor

assembler = VectorAssembler(
    inputCols=["dias_estimados", "dias_aprovacao", "dias_envio"],
    outputCol="features"
)

df = assembler.transform(df)

print("Vetor feito")

Vetor feito


In [12]:
train, test = df.randomSplit([0.8, 0.2], seed=42)

print("Treino e teste splitados")

Treino e teste splitados


In [14]:
rf = RandomForestClassifier(
    labelCol="atrasado",
    featuresCol="features",
    numTrees=80,
    maxDepth=6
    )
model = rf.fit(train)

print("Modelo treinado")

Modelo treinado


In [15]:
predictions = model.transform(test)

print("Previsões feitas")

Previsões feitas


In [16]:
evaluator = BinaryClassificationEvaluator(
    labelCol="atrasado",
    metricName="areaUnderROC"
)

auc = evaluator.evaluate(predictions)

print("AUC:", auc)

predictions.select("atrasado", "prediction", "probability").show(10)

AUC: 0.6731768695840475
+--------+----------+--------------------+
|atrasado|prediction|         probability|
+--------+----------+--------------------+
|       0|       0.0|[0.87592968086849...|
|       0|       0.0|[0.87592968086849...|
|       0|       0.0|[0.87592968086849...|
|       0|       0.0|[0.87561298866475...|
|       0|       0.0|[0.87561298866475...|
|       1|       0.0|[0.87561298866475...|
|       1|       0.0|[0.87153705163520...|
|       0|       0.0|[0.87320536508520...|
|       1|       0.0|[0.76314102086088...|
|       0|       0.0|[0.87592968086849...|
+--------+----------+--------------------+
only showing top 10 rows
